# ASL Recognition - Improved Transformer V2

Building on the original working approach with targeted improvements:
1. **Same fast data loading** as original
2. **Larger model**: More layers, more heads, wider dimensions
3. **Better features**: Position + Velocity + Acceleration
4. **Improved augmentation**: Temporal scaling, stronger noise
5. **Better training**: OneCycleLR, gradient accumulation, longer training

In [ ]:
import json
import os
import math
from typing import List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from torch.utils.data import Dataset, DataLoader, Sampler

from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Device: {device}, AMP: {use_amp}")

In [ ]:
# Config - same as original
BASE_PATH = r"/kaggle/input/asl-signs"
TRAIN_FILE = r"/kaggle/input/asl-signs/train.csv"
SIGN_INDEX_FILE = r"/kaggle/input/asl-signs/sign_to_prediction_index_map.json"

with open(SIGN_INDEX_FILE, "r") as f:
    SIGN2INDEX = json.load(f)

NUM_CLASSES = len(SIGN2INDEX)

INCLUDE_FACE = True
INCLUDE_DEPTH = False

FACE_LANDMARK_INDICES = {
    'nose': [1, 2, 4, 5, 6, 19, 94, 168, 197, 195],
    'left_eye': [33, 133, 160, 159, 158, 157, 173, 144, 145, 153, 154, 155, 156, 246, 7, 163],
    'right_eye': [263, 362, 387, 386, 385, 384, 398, 373, 374, 380, 381, 382, 466, 388, 390, 249],
    'left_eyebrow': [70, 63, 105, 66, 107, 55, 65, 52],
    'right_eyebrow': [300, 293, 334, 296, 336, 285, 295, 282],
    'mouth_outer': [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 409, 270, 269, 267, 0, 37, 39, 40, 185],
    'mouth_inner': [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95],
    'face_oval': [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 
                  152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
}

SELECTED_FACE_INDICES = []
for indices in FACE_LANDMARK_INDICES.values():
    SELECTED_FACE_INDICES.extend(indices)

print(f"Classes: {NUM_CLASSES}, Face landmarks: {len(SELECTED_FACE_INDICES)}")

In [ ]:
def generate_full_column_list() -> List[str]:
    landmark_specs = {"left_hand": 21, "pose": 33, "right_hand": 21}
    axes = ["x", "y", "z"] if INCLUDE_DEPTH else ["x", "y"]
    
    columns = []
    for lm_type, count in landmark_specs.items():
        for i in range(count):
            for ax in axes:
                columns.append(f"{lm_type}_{i}_{ax}")
    
    if INCLUDE_FACE:
        for face_idx in SELECTED_FACE_INDICES:
            for ax in axes:
                columns.append(f"face_{face_idx}_{ax}")
    
    return columns

ALL_COLUMNS = generate_full_column_list()
print(f"Total coordinate features: {len(ALL_COLUMNS)}")

In [ ]:
def normalize_values(df: pd.DataFrame) -> pd.DataFrame:
    axes = ["x", "y", "z"] if INCLUDE_DEPTH else ["x", "y"]
    origins = df[(df["type"] == "pose") & (df["landmark_index"] == 0)].set_index("frame")[axes]
    
    def normalize_frame(frame_df):
        frame = frame_df.name
        if frame in origins.index:
            frame_df[axes] = frame_df[axes] - origins.loc[frame]
        return frame_df
    
    return df.groupby("frame", group_keys=False).apply(normalize_frame)


def frame_stacked_data(file_path: str) -> np.ndarray:
    df = pd.read_parquet(os.path.join(BASE_PATH, file_path))
    
    if INCLUDE_FACE:
        face_df = df[df["type"] == "face"]
        face_df = face_df[face_df["landmark_index"].isin(SELECTED_FACE_INDICES)]
        other_df = df[df["type"] != "face"]
        df = pd.concat([face_df, other_df], ignore_index=True)
    else:
        df = df[df["type"] != "face"]
    
    axes = ["x", "y", "z"] if INCLUDE_DEPTH else ["x", "y"]
    df["UID"] = df["type"].astype(str) + "_" + df["landmark_index"].astype(str)
    df = df.sort_values(["frame", "UID"])
    df = normalize_values(df)
    
    pivot_df = df.pivot_table(index="frame", columns="UID", values=axes)
    pivot_df.columns = [f"{col[1]}_{col[0]}" for col in pivot_df.columns]
    pivot_df = pivot_df.reindex(columns=ALL_COLUMNS)
    
    return pivot_df.ffill().bfill().fillna(0).to_numpy().astype(np.float32)

In [ ]:
def augment_sample(x: np.ndarray) -> np.ndarray:
    """Enhanced augmentation"""
    x = x.copy()
    
    # Temporal scaling (speed variation) - IMPORTANT for sign recognition
    if np.random.random() > 0.3:
        scale = np.random.uniform(0.8, 1.2)
        T = x.shape[0]
        new_T = max(5, int(T * scale))
        indices = np.linspace(0, T - 1, new_T).astype(int)
        x = x[indices]
    
    # Gaussian noise
    if np.random.random() > 0.4:
        noise_std = np.random.uniform(0.002, 0.005)
        x = x + np.random.normal(0, noise_std, x.shape)
    
    # Spatial shift
    if np.random.random() > 0.5:
        shift = np.random.uniform(-0.03, 0.03, (1, x.shape[1]))
        x = x + shift
    
    # Temporal crop
    if np.random.random() > 0.5 and x.shape[0] > 20:
        max_crop = max(1, x.shape[0] // 8)
        start = np.random.randint(0, max_crop)
        end = x.shape[0] - np.random.randint(0, max_crop)
        if end > start + 5:
            x = x[start:end]
    
    # Random frame masking (dropout)
    if np.random.random() > 0.8:
        mask_ratio = np.random.uniform(0.05, 0.1)
        mask = np.random.random(x.shape[0]) > mask_ratio
        x = x * mask[:, np.newaxis]
    
    return x.astype(np.float32)

In [ ]:
class ASLDataset(Dataset):
    def __init__(self, videos, labels, max_frames=128, augment=False):
        self.videos = videos
        self.labels = labels
        self.max_frames = max_frames
        self.augment = augment
    
    def __len__(self):
        return len(self.videos)
    
    def __getitem__(self, idx):
        x = self.videos[idx]
        y = self.labels[idx]
        
        if self.augment:
            x = augment_sample(x)
        
        # Subsample if needed
        if x.shape[0] > self.max_frames:
            indices = np.linspace(0, x.shape[0] - 1, self.max_frames).astype(int)
            x = x[indices]
        
        # Compute velocity and acceleration
        vel = np.zeros_like(x)
        vel[1:] = x[1:] - x[:-1]
        
        acc = np.zeros_like(x)
        acc[1:] = vel[1:] - vel[:-1]
        
        # Concatenate: position + velocity + acceleration
        features = np.concatenate([x, vel, acc], axis=1)
        
        return torch.tensor(features, dtype=torch.float32), y

In [ ]:
def collate_fn(batch):
    seqs, labels = zip(*batch)
    
    lengths = [s.shape[0] for s in seqs]
    max_len = max(lengths)
    B, D = len(seqs), seqs[0].shape[1]
    
    padded = torch.zeros(B, max_len, D)
    mask = torch.zeros(B, max_len, dtype=torch.bool)
    
    for i, seq in enumerate(seqs):
        T = seq.shape[0]
        padded[i, :T] = seq
        mask[i, :T] = True
    
    return padded, mask, torch.tensor(labels)


class BucketSampler(Sampler):
    def __init__(self, lengths, batch_size, drop_last=False):
        self.lengths = lengths
        self.batch_size = batch_size
        self.drop_last = drop_last
    
    def __iter__(self):
        indices = np.argsort(self.lengths)
        batches = []
        for i in range(0, len(indices), self.batch_size):
            batch = indices[i:i + self.batch_size]
            if len(batch) == self.batch_size or not self.drop_last:
                batches.append(batch)
        np.random.shuffle(batches)
        for batch in batches:
            yield list(batch)
    
    def __len__(self):
        n = len(self.lengths)
        if self.drop_last:
            return n // self.batch_size
        return (n + self.batch_size - 1) // self.batch_size

In [ ]:
# ============== MODEL ==============

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(1)])

In [ ]:
class ImprovedTransformer(nn.Module):
    """Larger transformer with better architecture choices"""
    
    def __init__(self, input_dim, num_classes, d_model=256, n_heads=8, n_layers=6, dropout=0.1):
        super().__init__()
        
        # Input projection with layer norm
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout)
        )
        
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        
        # Pre-norm transformer (more stable training)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-norm
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        
        self.norm = nn.LayerNorm(d_model)
        
        # Deeper classification head
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x, mask):
        B, T, _ = x.shape
        
        x = self.input_proj(x)
        x = self.pos_enc(x)
        
        # Add CLS token
        cls = self.cls_token.expand(B, 1, -1)
        x = torch.cat([cls, x], dim=1)
        
        # Update mask for CLS
        cls_mask = torch.ones(B, 1, device=mask.device, dtype=torch.bool)
        mask = torch.cat([cls_mask, mask], dim=1)
        
        x = self.encoder(x, src_key_padding_mask=~mask)
        x = self.norm(x[:, 0])  # CLS token
        
        return self.head(x)

In [ ]:
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# ============== DATA LOADING ==============

def get_data_loaders(batch_size=64, max_frames=128):
    train_df = pd.read_csv(TRAIN_FILE)
    train_df['sign'] = train_df['sign'].map(SIGN2INDEX)
    
    train_split, val_split = train_test_split(
        train_df, test_size=0.1, stratify=train_df['sign'], random_state=42
    )
    
    print(f"Loading {len(train_split)} training videos...")
    train_videos = [frame_stacked_data(p) for p in train_split['path'].tolist()]
    
    print(f"Loading {len(val_split)} validation videos...")
    val_videos = [frame_stacked_data(p) for p in val_split['path'].tolist()]
    
    train_dataset = ASLDataset(train_videos, train_split['sign'].to_numpy(), max_frames, augment=True)
    val_dataset = ASLDataset(val_videos, val_split['sign'].to_numpy(), max_frames, augment=False)
    
    train_lengths = [min(v.shape[0], max_frames) for v in train_videos]
    val_lengths = [min(v.shape[0], max_frames) for v in val_videos]
    
    train_loader = DataLoader(
        train_dataset,
        batch_sampler=BucketSampler(train_lengths, batch_size),
        collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_dataset,
        batch_sampler=BucketSampler(val_lengths, batch_size),
        collate_fn=collate_fn
    )
    
    return train_loader, val_loader

In [ ]:
# Load data
BATCH_SIZE = 64
MAX_FRAMES = 128

train_loader, val_loader = get_data_loaders(BATCH_SIZE, MAX_FRAMES)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Get input dimension from first batch
sample_x, _, _ = next(iter(train_loader))
INPUT_DIM = sample_x.shape[-1]
print(f"Input dimension: {INPUT_DIM}")

In [ ]:
# ============== MODEL SETUP ==============

model = ImprovedTransformer(
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES,
    d_model=256,
    n_heads=8,
    n_layers=6,
    dropout=0.1
).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {num_params:,}")

In [ ]:
# Training config
EPOCHS = 120
PATIENCE = 25
ACCUMULATION_STEPS = 2  # Effective batch size = 64 * 2 = 128
MIXUP_PROB = 0.5
MIXUP_ALPHA = 0.2

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)

# OneCycleLR - often better than cosine annealing
scheduler = OneCycleLR(
    optimizer,
    max_lr=4e-4,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader) // ACCUMULATION_STEPS,
    pct_start=0.1,
    anneal_strategy='cos',
    div_factor=10,
    final_div_factor=100
)

scaler = GradScaler(enabled=use_amp)

In [ ]:
def train_epoch(loader):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for i, (x, mask, y) in enumerate(loader):
        x, mask, y = x.to(device), mask.to(device), y.to(device)
        
        with autocast(enabled=use_amp, device_type=device.type):
            # Mixup with probability
            if np.random.random() < MIXUP_PROB:
                x, y_a, y_b, lam = mixup_data(x, y, MIXUP_ALPHA)
                logits = model(x, mask)
                loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            else:
                logits = model(x, mask)
                loss = criterion(logits, y)
            
            loss = loss / ACCUMULATION_STEPS
        
        scaler.scale(loss).backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
        
        total_loss += loss.item() * ACCUMULATION_STEPS
    
    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch(loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for x, mask, y in loader:
        x, mask, y = x.to(device), mask.to(device), y.to(device)
        
        with autocast(enabled=use_amp, device_type=device.type):
            logits = model(x, mask)
            loss = criterion(logits, y)
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    
    return total_loss / len(loader), correct / total

In [ ]:
# ============== TRAINING ==============

best_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader)
    val_loss, val_acc = validate_epoch(val_loader)
    
    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Acc: {val_acc:.4f} | LR: {lr:.2e}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, 'best_model.pth')
        print(f"  -> New best! Acc: {val_acc:.4f}")
    else:
        patience_counter += 1
    
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nBest validation accuracy: {best_acc:.4f}")

In [ ]:
# Load best and evaluate
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

val_loss, val_acc = validate_epoch(val_loader)
print(f"Final validation accuracy: {val_acc:.4f}")